## LASSO REGRESSION

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, f1_score, accuracy_score
from sklearn.pipeline import Pipeline


In [ ]:

DATASET_ID = 3454

print(f"Fetching OpenML dataset {DATASET_ID}...")

try:
    # 'as_frame=False' is better for sparse data to ensure X is a matrix
    dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
    
    X_raw = dataset.data  
    y_raw = dataset.target 


    print(f"Dataset loaded: {dataset.details.get('name')}")
    print(f"Features shape: {X_raw.shape}")
    print(f"Target: {dataset.target_names[0]}") # Use .target_names for non-frame
    data_loaded = True

except Exception as e:
    print(f"Error loading dataset: {e}")
    data_loaded = False


 

In [ ]:
   
# Train-test split (on the RAW data) 
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Create the full processing and modeling pipeline
# This pipeline works perfectly with sparse matrices
lasso_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler(with_mean=False)), # with_mean=False is required for sparse
        ('lasso', LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=10000))
])


lasso_pipeline.fit(X_train, y_train)

y_pred = lasso_pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)


In [ ]:

print(f"\nModel Performance (Dataset: {DATASET_ID}):")
print(f"R² Score: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred)):.4f}")

# Identify selected features
lasso_model = lasso_pipeline.named_steps['lasso']
print(f"Optimal alpha: {lasso_model.alpha_:.6f}")

# Get feature names from the dataset object
    
feature_names = np.array(dataset.feature_names) 
selected_features_mask = np.abs(lasso_model.coef_) > 1e-6
selected_features = feature_names[selected_features_mask]
    
print(f"\nSelected {len(selected_features)} important features out of {X_raw.shape[1]} total.")

LASSO REGRESSION ON OPENML

In [ ]:

DATASET_ID = 3454

print(f"Fetching OpenML dataset {DATASET_ID}...")

try:
    # 'as_frame=False' is better for sparse data to ensure X is a matrix
    dataset = fetch_openml(data_id=DATASET_ID, as_frame=False, parser='auto')
    
    X_raw = dataset.data  
    y_raw = dataset.target 


    print(f"Dataset loaded: {dataset.details.get('name')}")
    print(f"Features shape: {X_raw.shape}")
    print(f"Target: {dataset.target_names[0]}") # Use .target_names for non-frame
    data_loaded = True

except Exception as e:
    print(f"Error loading dataset: {e}")
    data_loaded = False


In [ ]:
  
# Train-test split (on the RAW data) 
X_train, X_test, y_train, y_test = train_test_split(X_raw, y_raw, test_size=0.2, random_state=42)

# Create the full processing and modeling pipeline
# This pipeline works perfectly with sparse matrices
lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler(with_mean=False)), # with_mean=False is required for sparse
    ('lasso', LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=10000))
])


lasso_pipeline.fit(X_train, y_train)

# Evaluate 
y_pred = lasso_pipeline.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)


In [ ]:

print(f"\nModel Performance (Dataset: {DATASET_ID}):")
print(f"R² Score: {r2:.4f}")
print(f"MSE: {mse:.4f}")
print(f"MAE: {np.mean(np.abs(y_test - y_pred)):.4f}")

# Identify selected features
lasso_model = lasso_pipeline.named_steps['lasso']
print(f"Optimal alpha: {lasso_model.alpha_:.6f}")

# Get feature names from the dataset object
feature_names = np.array(dataset.feature_names) 
selected_features_mask = np.abs(lasso_model.coef_) > 1e-6
selected_features = feature_names[selected_features_mask]
    
print(f"\nSelected {len(selected_features)} important features out of {X_raw.shape[1]} total.")

# LASSO REGRESSION USING CSV

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.pipeline import Pipeline


Loading 'train1000.csv'...
Setup corrected for 'Option 1':
  Target (y): 'ptend_t_11'
  Features (X): 923 other columns (including 'ptend_t_0')
Data split into 800 train and 200 test samples.

--- Fitting LASSO Baseline (Experiment 1) ---

--- LASSO Results ---
R² Score:              0.9986
Mean Absolute Error:   0.0000
Mean Squared Error:    0.0000
Features Selected:     5 / 923


In [ ]:

# Load the dataset 
DATASET_FILE = 'train1000.csv'
print(f"Loading '{DATASET_FILE}'...")
data = pd.read_csv(DATASET_FILE).drop(columns=['sample_id', 'Unnamed: 0'], errors='ignore')


In [ ]:

# Coerce ALL columns to numeric right away
data = data.apply(pd.to_numeric, errors='coerce')

NEW_TARGET_NAME = 'ptend_t_11' 

# Check if the new target is in the file
if NEW_TARGET_NAME not in data.columns:
    print(f"Error: Target '{NEW_TARGET_NAME}' not found in columns.")
    print("Please check the name from your correlation screenshot.")
else:
    # Create the new valid X and y
    y_raw = data[NEW_TARGET_NAME]
    X_raw = data.drop(columns=[NEW_TARGET_NAME])

    print(f"Setup corrected for 'Option 1':")
    print(f"  Target (y): '{NEW_TARGET_NAME}'")
    print(f"  Features (X): {X_raw.shape[1]} other columns (including 'ptend_t_0')")

    #  Split the raw data
    X_train, X_test, y_train, y_test = train_test_split(
        X_raw, y_raw, test_size=0.2, random_state=42
    )
    print(f"Data split into {len(X_train)} train and {len(X_test)} test samples.")


In [ ]:

#  Create the LASSO pipeline 
lasso_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(cv=5, random_state=42, n_jobs=-1, max_iter=10000))
])

print("\nFitting LASSO Baseline")
lasso_pipeline.fit(X_train, y_train)

#Evaluate
y_pred_lasso = lasso_pipeline.predict(X_test)
r2_lasso = r2_score(y_test, y_pred_lasso)
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
mse_lasso = mean_squared_error(y_test, y_pred_lasso) 

# Get results
lasso_model = lasso_pipeline.named_steps['lasso']
total_features = X_train.shape[1]
selected_features_lasso = np.sum(np.abs(lasso_model.coef_) > 1e-6)


In [ ]:

print(f"\n LASSO Results ")
print(f"R² Score:              {r2_lasso:.4f}")
print(f"Mean Absolute Error:   {mae_lasso:.4f}")
print(f"Mean Squared Error:    {mse_lasso:.4f}") 
print(f"Features Selected:     {selected_features_lasso} / {total_features}")

# DEXTER - LASSO

In [ ]:
# Load Dataset
dataset = openml.datasets.get_dataset(4136)

X, y, _, _ = dataset.get_data(
    dataset_format="dataframe",
    target=dataset.default_target_attribute
)


c:\Users\shrey\Desktop\RAVEN\venv\Lib\site-packages\openml\datasets\dataset.py:472: FutureWarning: factorize with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  pd.factorize(type_)[0]


Number of redundant (zero-weight) features: 19740
Number of selected features: 260

--- TRUE LASSO Classification Results (L1 Logistic Regression) ---
Accuracy: 91.67%
F1 Score: 0.9153


In [ ]:

# Dexter classes are {-1, 1}; convert to {0, 1}
le = LabelEncoder()
y_enc = le.fit_transform(y)

# =Train/Test Split 
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.3, random_state=42
)

# LASSO CLASSIFICATION (L1 Logistic Regression)
lasso_clf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler(with_mean=False)),
    ('logreg', LogisticRegression(
        penalty='l1',
        solver='liblinear',   
        max_iter=5000,
        C=1.0,                
        random_state=42
    ))
])

lasso_clf.fit(X_train, y_train)

# Predictions
y_pred = lasso_clf.predict(X_test)
coef = lasso_clf.named_steps['logreg'].coef_[0]  # shape: (n_features,)

# Count features with zero coefficient (redundant features)
n_redundant = np.sum(coef == 0)


n_selected = np.sum(coef != 0)


In [ ]:

print(f"Number of redundant (zero-weight) features: {n_redundant}")
print(f"Number of selected features: {n_selected}")

# Evaluation
print("\n LASSO  Results (L1 Logistic Regression) ")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"F1 Score: {f1_score(y_test, y_pred):.4f}")
